# Airborne Survey Flight Path Planner

This notebook generates a dogbone-style flight path for an airborne survey using a King Air B200 at 28,000 ft and 200 knots. The survey uses a 5 km swath width camera and covers an area defined by GPS coordinates in decimal degrees. The output includes a map overlay of the flight lines and coverage polygon.

In [125]:
import sys
import subprocess
import importlib.util

required_packages = [
    'numpy',
    'shapely',
    'pyproj',
    'folium'
]
for package in required_packages:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])

print('Required packages are installed.')

Required packages are installed.


In [126]:
import csv
import math
import numpy as np
from shapely.geometry import LineString, Polygon, MultiLineString, GeometryCollection, Point, mapping
from shapely.ops import unary_union, transform as shapely_transform
from shapely import affinity
from pyproj import CRS, Transformer # a pythonic Coordinate Reference System and transformation library
import folium
from IPython.display import HTML

# Survey parameters
# service_altitude_ft = 28000  # aircraft altitude
groundspeed_kt = 200     # knots
swath_width_km = 10       # camera swath width in kilometers
swath_overlap = 0.1        # optional overlap fraction between adjacent passes
perimeter_margin_km = 5.0  # add extra coverage around the survey boundary
initial_heading_deg = 30  # aircraft heading in degrees true north (0-360)
lat_offset = 0.1    # latitude offset in decimal degrees
lon_offset = -0.05    # longitude offset in decimal degrees

# Example area coordinates in decimal degrees (latitude, longitude).
# This can be any convex boundary, including a rectangle rotated diagonally.
area_name = "LaRC_Area"
waypoint_prefix = 'LRC'
survey_boundary = [
    (37.165, -76.701, 'PowerPlant'),
    (36.943, -76.329, 'SWPV2_buoy'),
    (37.201, -76.266, '44072_buoy'),
    (37.227, -76.479, 'EYKTV2_buoy'),
    (37.567, -76.257, '44058_buoy')
]

# Helper transformers for metric coordinates
crs_geo = CRS.from_epsg(4326)
survey_center_lat = np.mean([lat for lat, lon, *_ in survey_boundary])
survey_center_lon = np.mean([lon for lat, lon, *_ in survey_boundary])
utm_zone = int((survey_center_lon + 180.0) // 6.0) + 1
epsg_code = 32600 + utm_zone if survey_center_lat >= 0 else 32700 + utm_zone
crs_local = CRS.from_epsg(epsg_code)  # local UTM projection for accurate metric spacing

to_m = Transformer.from_crs(crs_geo, crs_local, always_xy=True)
to_geo = Transformer.from_crs(crs_local, crs_geo, always_xy=True)


def _normalize_boundary_points(latlon_coords):
    normalized = []
    for point in latlon_coords:
        if len(point) >= 2:
            lat = point[0]
            lon = point[1]
            name = point[2] if len(point) > 2 else None
            normalized.append((lat, lon, name))
    return normalized


def latlon_to_xy(lat_lon_points):
    normalized = _normalize_boundary_points(lat_lon_points)
    return [to_m.transform(lon, lat) for lat, lon, _ in normalized]


def xy_to_latlon(xy_points):
    return [(to_geo.transform(x, y)[1], to_geo.transform(x, y)[0]) for x, y in xy_points]


def dd_to_honeywell_format(value, positive_indicator, negative_indicator):
    sign = positive_indicator if value >= 0 else negative_indicator
    abs_value = abs(value)
    degrees = int(abs_value)
    minutes = (abs_value - degrees) * 60.0
    return f"{sign} {degrees:02d} {minutes:05.2f} "


def summarize_segment_travel(flight_pattern, groundspeed_kt=200.0):
    coords = list(flight_pattern.coords)
    if len(coords) < 2:
        return [], 0.0, 0.0, 0.0

    segment_summaries = []
    total_distance_m = 0.0
    for idx in range(1, len(coords)):
        x1, y1 = coords[idx - 1]
        x2, y2 = coords[idx]
        dist_m = math.hypot(x2 - x1, y2 - y1)
        dist_nm = dist_m / 1852.0
        travel_min = (dist_nm / groundspeed_kt) * 60.0 if groundspeed_kt > 0 else float('nan')
        segment_summaries.append((idx, dist_m, dist_nm, travel_min))
        total_distance_m += dist_m

    total_distance_nm = total_distance_m / 1852.0
    total_time_min = (total_distance_nm / groundspeed_kt) * 60.0 if groundspeed_kt > 0 else float('nan')
    return segment_summaries, total_distance_m, total_distance_nm, total_time_min


def build_rectangular_pattern(latlon_coords, swath_km=5.0, overlap=0.1, perimeter_margin_km=5.0, initial_heading_deg=45.0, lat_offset=0.0, lon_offset=0.0):
    if len(latlon_coords) < 3:
        raise ValueError('At least three coordinates are required to define a survey area.')

    normalized_points = _normalize_boundary_points(latlon_coords)
    input_xy = latlon_to_xy(normalized_points)
    survey_poly = Polygon(input_xy).buffer(0)

    if survey_poly.is_empty or survey_poly.area == 0:
        raise ValueError('Survey polygon area is zero. Check input coordinates.')

    if perimeter_margin_km > 0:
        survey_poly = survey_poly.buffer(perimeter_margin_km * 1000.0)

    # Rotate the survey polygon so the requested heading becomes horizontal,
    # then build an axis-aligned rectangular bounding box around it.
    heading_angle_deg = (90.0 - initial_heading_deg) % 360.0
    rotated_poly = affinity.rotate(survey_poly, -heading_angle_deg, origin='centroid', use_radians=False)
    minx, miny, maxx, maxy = rotated_poly.bounds
    rect_coords = [(minx, miny), (maxx, miny), (maxx, maxy), (minx, maxy), (minx, miny)]
    survey_rect = Polygon(rect_coords)
    survey_rect = affinity.rotate(survey_rect, heading_angle_deg, origin='centroid', use_radians=False)

    if lat_offset != 0.0 or lon_offset != 0.0:
        lat_offset_m = lat_offset * 111320.0
        lon_offset_m = lon_offset * 111320.0 * math.cos(math.radians(survey_center_lat))
        survey_rect = affinity.translate(survey_rect, xoff=lon_offset_m, yoff=lat_offset_m)

    rotated_rect = affinity.rotate(survey_rect, -heading_angle_deg, origin='centroid', use_radians=False)
    minx, miny, maxx, maxy = rotated_rect.bounds
    center_y = (miny + maxy) / 2.0
    line_spacing = swath_km * 1000 * (1 - overlap)

    pass_segments = []
    height = maxy - miny
    if height <= line_spacing:
        candidate_ys = [center_y]
    else:
        num_lines = int(math.floor(height / line_spacing)) + 1
        start_y = center_y - ((num_lines - 1) * line_spacing / 2.0)
        candidate_ys = [start_y + i * line_spacing for i in range(num_lines)]

    for current_y in candidate_ys:
        pass_line = LineString([(minx - 10000, current_y), (maxx + 10000, current_y)])
        clipped = pass_line.intersection(rotated_rect)

        if clipped.is_empty or getattr(clipped, 'geom_type', None) not in {'LineString', 'MultiLineString', 'GeometryCollection'}:
            continue

        if isinstance(clipped, LineString):
            if clipped.length > 0:
                pass_segments.append(clipped)
        elif isinstance(clipped, MultiLineString):
            pass_segments.extend([seg for seg in clipped.geoms if seg.length > 0])
        elif isinstance(clipped, GeometryCollection):
            for part in clipped.geoms:
                if isinstance(part, LineString) and part.length > 0:
                    pass_segments.append(part)

    if not pass_segments:
        center_y = (miny + maxy) / 2.0
        pass_line = LineString([(minx - 10000, center_y), (maxx + 10000, center_y)])
        clipped = pass_line.intersection(rotated_rect)
        if isinstance(clipped, LineString) and clipped.length > 0:
            pass_segments.append(clipped)
        elif isinstance(clipped, MultiLineString):
            pass_segments.extend([seg for seg in clipped.geoms if seg.length > 0])

    if not pass_segments:
        raise ValueError('No pass segments could be generated. Adjust the survey area or swath width.')

    pass_segments = sorted(pass_segments, key=lambda s: s.centroid.y)
    pattern_points = []
    for idx, segment in enumerate(pass_segments):
        coords = list(segment.coords)
        if idx % 2 == 1:
            coords = coords[::-1]
        if pattern_points and pattern_points[-1] != coords[0]:
            pattern_points.append(coords[0])
        pattern_points.extend(coords)

    pattern_line = LineString(pattern_points)
    rotated_pattern = affinity.rotate(pattern_line, heading_angle_deg, origin='centroid', use_radians=False)
    return survey_rect, rotated_pattern, pass_segments, initial_heading_deg


def export_foreflight_waypoints(flight_pattern, filename=None):
    if filename is None:
        filename = f"{area_name.replace(' ', '_')}_waypoints_foreflight.csv"
    latlon_points = xy_to_latlon(list(flight_pattern.coords))
    deduped = []
    for point in latlon_points:
        if not deduped or deduped[-1] != point:
            deduped.append(point)

    with open(filename, 'w', newline='') as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(['Waypoint', 'Description', 'LAT', 'LONG'])
        for idx, (lat, lon) in enumerate(deduped, start=1):
            writer.writerow([waypoint_prefix+f'{idx:02d}', 'NA', f'{lat:.4f}', f'{lon:.4f}'])

    print(f'Wrote {len(deduped)} waypoints to {filename}')
    return filename


def export_honeywell_fms_waypoints(flight_pattern, filename=None):
    if filename is None:
        filename = f"{area_name.replace(' ', '_')}_waypoints_honeywell.csv"
    latlon_points = xy_to_latlon(list(flight_pattern.coords))
    deduped = []
    for point in latlon_points:
        if not deduped or deduped[-1] != point:
            deduped.append(point)

    with open(filename, 'w', newline='') as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(['E', 'WPT', 'FIX', 'LAT', 'LON'])
        for idx, (lat, lon) in enumerate(deduped, start=1):
            lat_fmt = dd_to_honeywell_format(lat, 'N', 'S')
            lon_fmt = dd_to_honeywell_format(lon, 'E', 'W')
            writer.writerow(['X', waypoint_prefix+f'{idx:02d}', 'NA', lat_fmt, lon_fmt])

    print(f'Wrote {len(deduped)} Honeywell waypoints to {filename}')
    return filename


survey_poly, survey_pattern, segments, orientation = build_rectangular_pattern(
    survey_boundary,
    swath_width_km,
    swath_overlap,
    perimeter_margin_km=perimeter_margin_km,
    initial_heading_deg=initial_heading_deg,
    lat_offset=lat_offset,
    lon_offset=lon_offset
)
waypoint_csv = export_foreflight_waypoints(survey_pattern)
honeywell_csv = export_honeywell_fms_waypoints(survey_pattern)

segment_summaries, total_distance_m, total_distance_nm, total_time_min = summarize_segment_travel(survey_pattern, groundspeed_kt)
print(f'Survey area polygon uses {len(survey_boundary)} input points.')
print(f'Generated {len(segments)} survey passes with {swath_width_km} km swath, {swath_overlap*100:.0f}% overlap, and {perimeter_margin_km:.0f} km perimeter margin.')
print(f'Pattern heading (degrees true north): {orientation:.1f}')
print(f'Total flight path length: {total_distance_m:.2f} m ({total_distance_nm:.3f} nm)')
print(f'Estimated travel time at {groundspeed_kt:.0f} kt: {total_time_min:.1f} min')
for seg_idx, dist_m, dist_nm, travel_min in segment_summaries:
    print(f'Segment {seg_idx}: {dist_m/1000:.2f} km ({dist_nm:.2f} nm), {travel_min:.1f} min')
print(f'ForeFlight waypoint file: {waypoint_csv}')
print(f'Honeywell FMS waypoint file: {honeywell_csv}')

Wrote 12 waypoints to LaRC_Area_waypoints_foreflight.csv
Wrote 12 Honeywell waypoints to LaRC_Area_waypoints_honeywell.csv
Survey area polygon uses 5 input points.
Generated 6 survey passes with 10 km swath, 10% overlap, and 5 km perimeter margin.
Pattern heading (degrees true north): 30.0
Total flight path length: 486250.25 m (262.554 nm)
Estimated travel time at 200 kt: 78.8 min
Segment 1: 73.54 km (39.71 nm), 11.9 min
Segment 2: 9.00 km (4.86 nm), 1.5 min
Segment 3: 0.00 km (0.00 nm), 0.0 min
Segment 4: 73.54 km (39.71 nm), 11.9 min
Segment 5: 9.00 km (4.86 nm), 1.5 min
Segment 6: 0.00 km (0.00 nm), 0.0 min
Segment 7: 73.54 km (39.71 nm), 11.9 min
Segment 8: 9.00 km (4.86 nm), 1.5 min
Segment 9: 0.00 km (0.00 nm), 0.0 min
Segment 10: 73.54 km (39.71 nm), 11.9 min
Segment 11: 9.00 km (4.86 nm), 1.5 min
Segment 12: 0.00 km (0.00 nm), 0.0 min
Segment 13: 73.54 km (39.71 nm), 11.9 min
Segment 14: 9.00 km (4.86 nm), 1.5 min
Segment 15: 0.00 km (0.00 nm), 0.0 min
Segment 16: 73.54 km (39.

In [127]:
def build_map(latlon_coords, survey_poly, flight_pattern, pass_segments, map_filename=None):
    from shapely.geometry import mapping
    from shapely.ops import transform as shapely_transform

    if map_filename is None:
        map_filename = f"{area_name.replace(' ', '_')}_flight_path.html"
    center_lat = np.mean([lat for lat, lon, *_ in latlon_coords])
    center_lon = np.mean([lon for lat, lon, *_ in latlon_coords])
    survey_map = folium.Map(location=[center_lat, center_lon], zoom_start=9, tiles='OpenStreetMap')

    hull_latlon = xy_to_latlon(list(survey_poly.exterior.coords))
    folium.PolyLine(hull_latlon, color='blue', weight=3, opacity=0.7, tooltip='Survey boundary').add_to(survey_map)

    for idx, point in enumerate(latlon_coords, start=1):
        lat = point[0]
        lon = point[1]
        wp_name = point[2] if len(point) > 2 else f"WP{idx:02d}"
        folium.Marker(
            location=[lat, lon],
            popup=f"{wp_name}: ({lat:.6f}, {lon:.6f})",
            tooltip=f"Survey waypoint {wp_name}",
            icon=folium.Icon(color='purple', icon='info-sign')
        ).add_to(survey_map)

    pattern_latlon = xy_to_latlon(list(flight_pattern.coords))
    folium.PolyLine(pattern_latlon, color='red', weight=4, opacity=0.9, dash_array='5, 6', tooltip='Rectangular survey flight path').add_to(survey_map)

    """
    if pass_segments:
        half_swath_m = swath_width_km * 1000.0 / 2.0
        coverage_geom = unary_union([
            segment.buffer(half_swath_m, cap_style='flat', join_style='mitre')
            for segment in pass_segments
        ])
        if not coverage_geom.is_empty:
            coverage_geom_geo = shapely_transform(to_geo.transform, coverage_geom)
            folium.GeoJson(
                mapping(coverage_geom_geo),
                style_function=lambda feature: {
                    'fillColor': 'orange',
                    'color': 'orange',
                    'weight': 2,
                    'fillOpacity': 0.25,
                },
                tooltip='Actual survey coverage (half swath on each side of flight path)'
            ).add_to(survey_map)
    """

    survey_map.save(map_filename)
    return survey_map, map_filename

survey_map, map_filename = build_map(survey_boundary, survey_poly, survey_pattern, segments)
survey_map
print(f"Map file: {map_filename}")

Map file: LaRC_Area_flight_path.html


In [128]:
# Compute and print flight path length
try:
    length_m = survey_pattern.length
    length_km = length_m / 1000.0
    length_nm = length_m / 1852.0
    print(f"Flight path length: {length_m:.2f} m ({length_km:.3f} km, {length_nm:.3f} nmi)")
except NameError:
    print('`survey_pattern` not found in kernel. Recomputing pattern...')
    survey_poly, survey_pattern, segments, orientation = build_rectangular_pattern(survey_boundary, swath_width_km, swath_overlap)
    length_m = survey_pattern.length
    length_km = length_m / 1000.0
    length_nm = length_m / 1852.0
    print(f"Flight path length: {length_m:.2f} m ({length_km:.3f} km, {length_nm:.3f} nmi)")

# Show the generated CSV file names
try:
    print(f"ForeFlight CSV: {waypoint_csv}")
    print(f"Honeywell FMS CSV: {honeywell_csv}")
except NameError:
    print('Waypoint CSVs not generated yet. Run the pattern cell first.')


Flight path length: 486250.25 m (486.250 km, 262.554 nmi)
ForeFlight CSV: LaRC_Area_waypoints_foreflight.csv
Honeywell FMS CSV: LaRC_Area_waypoints_honeywell.csv
